# Contrastive Evidence Fusion — NIST-Trained, Orbitrap-Evaluated

## Design

**Stage 1 — Contrastive pretraining** (NIST, no labels):
Learn spectral representations from 20K NIST23 gold-standard spectra.
Same compound (IK14) = pull together, different compound = push apart.

**Stage 2 — Confidence head** (NIST leave-one-out, clean labels):
For each NIST query, search candidates within 10 ppm. Train a fusion model:
`[query_embed, cand_embed, cosine_sim, scalar_features] → P(correct match)`
No RT available — model learns from MS2 + mass accuracy only.

**Stage 3 — RT calibration** (500 Oliver spectra, Option B):
Thin calibration layer: `[NIST_confidence, delta_rt] → P(correct)`
Only 500 spectra from Oliver — just enough to learn "how much does RT matter?"

**Evaluation**: remaining ~5,100 Oliver spectra (never seen during training or calibration).

In [ ]:
import sys, json, time, warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from bisect import bisect_left, bisect_right
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import ms_entropy as me

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

N_BINS = 500
EMBED_DIM = 64
PPM_TOL = 10
MS2_TOL = 0.05

print('Ready')

## Parse NIST + bin spectra

In [ ]:
def parse_msp(path):
    spectra = []
    current = {}
    peaks = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                if current and peaks:
                    arr = np.array(peaks, dtype=np.float32)
                    base = arr[:, 1].max()
                    arr = arr[arr[:, 1] >= 0.01 * base]
                    if len(arr) > 0:
                        current['peaks'] = arr
                        spectra.append(current)
                current = {}; peaks = []
                continue
            if ':' in line and not line[0].isdigit():
                key, val = line.split(':', 1)
                key = key.strip().lower(); val = val.strip()
                if key == 'name': current['name'] = val
                elif key == 'precursormz': current['precursor_mz'] = float(val)
                elif key == 'inchikey': current['ik14'] = val[:14]
            elif line[0].isdigit():
                parts = line.split()
                if len(parts) >= 2:
                    peaks.append([float(parts[0]), float(parts[1])])
    if current and peaks:
        arr = np.array(peaks, dtype=np.float32)
        arr = arr[arr[:, 1] >= 0.01 * arr[:, 1].max()]
        if len(arr) > 0:
            current['peaks'] = arr
            spectra.append(current)
    return spectra

def bin_spectrum(peaks, n_bins=N_BINS):
    vec = np.zeros(n_bins, dtype=np.float32)
    if peaks is None or len(peaks) == 0: return vec
    for mz, intensity in peaks:
        b = int(mz)
        if 0 <= b < n_bins: vec[b] += intensity
    total = vec.sum()
    if total > 0: vec /= total
    return vec

t0 = time.time()
nist = parse_msp('../data/reference_db/nist_protonated_sampled.msp')
print('Parsed %d NIST spectra in %.1fs' % (len(nist), time.time()-t0))

# Sort by precursor m/z for efficient LOO search
nist.sort(key=lambda s: s['precursor_mz'])
mz_array = np.array([s['precursor_mz'] for s in nist])

# Bin all spectra
t0 = time.time()
nist_binned = np.stack([bin_spectrum(s['peaks']) for s in nist])
print('Binned: %s in %.1fs' % (str(nist_binned.shape), time.time()-t0))

# Clean peaks for entropy similarity (remove precursor, apply weighting)
for s in nist:
    mask = np.abs(s['peaks'][:, 0] - s['precursor_mz']) > MS2_TOL
    s['peaks_clean'] = s['peaks'][mask]

# IK14 stats
ik14s = [s.get('ik14', '') for s in nist]
unique_ik14 = set(ik14s) - {''}
print('Unique compounds (IK14): %d' % len(unique_ik14))

# Group by IK14 for contrastive pairs
ik14_to_idx = defaultdict(list)
for i, ik in enumerate(ik14s):
    if ik: ik14_to_idx[ik].append(i)
pair_ik14s = {k: v for k, v in ik14_to_idx.items() if len(v) >= 2}
n_pair_spectra = sum(len(v) for v in pair_ik14s.values())
print('IK14s with 2+ spectra: %d (%d spectra)' % (len(pair_ik14s), n_pair_spectra))

## Stage 1 — Contrastive pretraining on NIST

In [ ]:
class SpectralEncoder(nn.Module):
    def __init__(self, input_dim=N_BINS, embed_dim=EMBED_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, embed_dim),
        )
    def forward(self, x):
        return F.normalize(self.encoder(x), p=2, dim=-1)

class ContrastivePairDataset(Dataset):
    def __init__(self, spectra_matrix, labels_list, groups_dict):
        self.spectra = spectra_matrix
        self.anchors = []
        for ik, idxs in groups_dict.items():
            for idx in idxs:
                self.anchors.append((idx, ik))
        self.groups = groups_dict
    def __len__(self): return len(self.anchors)
    def __getitem__(self, i):
        ai, ik = self.anchors[i]
        group = self.groups[ik]
        pi = ai
        while pi == ai: pi = group[np.random.randint(len(group))]
        return torch.tensor(self.spectra[ai]), torch.tensor(self.spectra[pi])

def info_nce_loss(anchors, positives, temperature=0.07):
    sim = torch.mm(anchors, positives.t()) / temperature
    return F.cross_entropy(sim, torch.arange(sim.shape[0]))

# ── Pretrain ──
torch.manual_seed(42)
encoder = SpectralEncoder()
ds = ContrastivePairDataset(nist_binned, ik14s, pair_ik14s)
loader = DataLoader(ds, batch_size=512, shuffle=True, drop_last=True)
opt = optim.Adam(encoder.parameters(), lr=1e-3, weight_decay=1e-5)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)

print('Contrastive pretraining: %d anchors, %d compounds' % (len(ds), len(pair_ik14s)))
t0 = time.time()
for epoch in range(30):
    encoder.train()
    eloss = 0; nb = 0
    for ab, pb in loader:
        loss = info_nce_loss(encoder(ab), encoder(pb))
        opt.zero_grad(); loss.backward(); opt.step()
        eloss += loss.item(); nb += 1
    sched.step()
    if (epoch+1) % 10 == 0:
        print('  Epoch %d: loss=%.4f' % (epoch+1, eloss/nb))
print('Done in %.1fs' % (time.time()-t0))

pretrained_state = {k: v.clone() for k, v in encoder.state_dict().items()}

## Stage 2 — NIST leave-one-out: build pairs and train confidence head

For each NIST spectrum, search for candidates within 10 ppm. Compute:
- Spectral embeddings for query and candidate (from pretrained encoder)
- Cosine similarity in embedding space (learned similarity)
- Entropy similarity (traditional similarity — for comparison)
- delta_mda, n_candidates

Train a fusion head on these pairs. No RT — NIST doesn't have it.

In [ ]:
# ── Leave-one-out search: build (query, candidate) pairs ──
# This re-runs the LOO but stores query_idx AND cand_idx for spectral encoding

t0 = time.time()
n = len(nist)
rows = []

for qi in range(n):
    q = nist[qi]
    qmz = q['precursor_mz']
    qik = q.get('ik14', '')
    qpeaks = q['peaks_clean']

    if len(qpeaks) == 0 or not qik:
        continue

    lo = bisect_left(mz_array, qmz * (1 - PPM_TOL / 1e6))
    hi = bisect_right(mz_array, qmz * (1 + PPM_TOL / 1e6))

    for ci in range(lo, hi):
        if ci == qi: continue
        c = nist[ci]
        if len(c['peaks_clean']) == 0: continue

        esim = me.calculate_entropy_similarity(
            qpeaks, c['peaks_clean'], ms2_tolerance_in_da=MS2_TOL, clean_spectra=True)

        rows.append({
            'query_idx': qi,
            'cand_idx': ci,
            'entropy_sim': esim,
            'delta_mda': abs(qmz - c['precursor_mz']) * 1000,
            'correct': int(c.get('ik14', '') == qik and qik != ''),
        })

    if (qi + 1) % 5000 == 0:
        print('  %d/%d queries (%.1fs, %d pairs)' % (qi+1, n, time.time()-t0, len(rows)))

pairs = pd.DataFrame(rows)
print('\nLOO done in %.1fs' % (time.time()-t0))
print('Pairs: %d' % len(pairs))
print('Queries: %d' % pairs['query_idx'].nunique())
print('Correct: %d (%.1f%%)' % (pairs['correct'].sum(), pairs['correct'].mean()*100))

# Add n_candidates and esim_rank per query
n_cands = pairs.groupby('query_idx').size().rename('n_candidates')
pairs = pairs.merge(n_cands.reset_index(), on='query_idx')
pairs['esim_rank'] = pairs.groupby('query_idx')['entropy_sim'].rank(ascending=False).astype(int)

In [ ]:
# ── Compute spectral embeddings for all pairs ──
encoder.eval()
with torch.no_grad():
    all_emb = encoder(torch.tensor(nist_binned)).numpy()  # (20K, 64)

# For each pair: cosine similarity in embedding space
q_emb = all_emb[pairs['query_idx'].values]
c_emb = all_emb[pairs['cand_idx'].values]
pairs['emb_cosine'] = (q_emb * c_emb).sum(axis=1)  # already L2-normalized

# Also compute element-wise difference norm
pairs['emb_dist'] = np.linalg.norm(q_emb - c_emb, axis=1)

print('Embedding features computed')
print('emb_cosine — TP mean: %.3f  FP mean: %.3f' % (
    pairs.loc[pairs['correct']==1, 'emb_cosine'].mean(),
    pairs.loc[pairs['correct']==0, 'emb_cosine'].mean()))
print('entropy_sim — TP mean: %.3f  FP mean: %.3f' % (
    pairs.loc[pairs['correct']==1, 'entropy_sim'].mean(),
    pairs.loc[pairs['correct']==0, 'entropy_sim'].mean()))

# Pair-level AUC comparison
auc_esim = roc_auc_score(pairs['correct'], pairs['entropy_sim'])
auc_emb = roc_auc_score(pairs['correct'], pairs['emb_cosine'])
auc_dist = roc_auc_score(pairs['correct'], -pairs['emb_dist'])
print('\nPair-level AUC:')
print('  entropy_sim:    %.3f' % auc_esim)
print('  emb_cosine:     %.3f  (contrastive learned similarity)' % auc_emb)
print('  -emb_dist:      %.3f' % auc_dist)

## Train confidence head on NIST (no RT)

Features for the fusion model:
- `emb_cosine`: learned spectral similarity in embedding space
- `emb_dist`: Euclidean distance in embedding space
- `entropy_sim`: traditional entropy similarity (baseline feature)
- `delta_mda`: mass accuracy
- `n_candidates`: how crowded is the mass window
- `esim_rank`: rank of this candidate by entropy_sim

GroupKFold on query_idx — same evaluation as fdr_benchmark_composite.

In [ ]:
NIST_FEAT_COLS = ['emb_cosine', 'emb_dist', 'entropy_sim', 'delta_mda', 'n_candidates', 'esim_rank']

X_nist = pairs[NIST_FEAT_COLS].values.astype(np.float32)
y_nist = pairs['correct'].values
g_nist = pairs['query_idx'].values

gkf = GroupKFold(n_splits=5)

# Models: GBM (scalar only) vs GBM (with embedding features)
SCALAR_ONLY = ['entropy_sim', 'delta_mda', 'n_candidates', 'esim_rank']

oof_nist = {
    'esim_only':   np.full(len(pairs), np.nan),
    'gbm_scalar':  np.full(len(pairs), np.nan),
    'gbm_emb':     np.full(len(pairs), np.nan),
}

for fold, (tr, te) in enumerate(gkf.split(X_nist, y_nist, g_nist)):
    # Entropy sim alone (no training needed)
    oof_nist['esim_only'][te] = pairs.iloc[te]['entropy_sim'].values

    # GBM on scalar features only
    X_sc = pairs[SCALAR_ONLY].values.astype(np.float32)
    gbm_sc = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                         subsample=0.8, random_state=42)
    gbm_sc.fit(X_sc[tr], y_nist[tr])
    oof_nist['gbm_scalar'][te] = gbm_sc.predict_proba(X_sc[te])[:, 1]

    # GBM with embedding features
    gbm_emb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                          subsample=0.8, random_state=42)
    gbm_emb.fit(X_nist[tr], y_nist[tr])
    oof_nist['gbm_emb'][te] = gbm_emb.predict_proba(X_nist[te])[:, 1]

# ── NIST pair-level AUC ──
print('=== NIST Leave-One-Out: Pair-Level AUC ===')
for name, label in [('esim_only', 'Entropy sim only'),
                    ('gbm_scalar', 'GBM (scalar, no embedding)'),
                    ('gbm_emb', 'GBM (scalar + embedding)')]:
    v = ~np.isnan(oof_nist[name])
    auc = roc_auc_score(y_nist[v], oof_nist[name][v])
    print('  %-30s  AUC=%.4f' % (label, auc))

# ── NIST query-level: does the correct candidate rank #1? ──
print('\n=== NIST Query-Level: Top-1 Accuracy ===')
for name, label in [('esim_only', 'Entropy sim'),
                    ('gbm_scalar', 'GBM scalar'),
                    ('gbm_emb', 'GBM + embedding')]:
    pairs['_score'] = oof_nist[name]
    top1_nist = pairs.sort_values('_score', ascending=False).groupby('query_idx').first()
    acc = top1_nist['correct'].mean()
    print('  %-30s  Top-1 acc=%.1f%%' % (label, acc * 100))

# ── Train final GBM+embedding model on ALL NIST data for Stage 3 ──
gbm_nist_final = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)
gbm_nist_final.fit(X_nist, y_nist)
nist_scaler = StandardScaler().fit(X_nist)  # save scaler for Orbitrap application
print('\nFinal NIST model trained on all %d pairs' % len(pairs))

## Stage 3 — Apply to Oliver's Orbitrap + RT calibration

Apply the NIST-trained model to Oliver's data to get a "spectral confidence" score.
Then calibrate with RT using 500 Oliver spectra (Option B). Evaluate on the remaining ~5,100.

The spectral confidence captures everything the model learned from NIST.
The RT calibration adds the one feature NIST couldn't provide.

In [ ]:
# ── Load Oliver's Orbitrap data ──
ft_orbi = pd.read_csv('../data/feature_table_v2.csv')
top1_orbi = (ft_orbi.sort_values('entropy_similarity', ascending=False)
             .groupby('wiki_id').first().reset_index())
labels_orbi = top1_orbi['hit_label'].values

with open('../data/library_peaks_cache.json') as f:
    lib_peaks = json.load(f)

# Bin reference spectra for Orbitrap top-1
ref_binned_orbi = np.zeros((len(top1_orbi), N_BINS), dtype=np.float32)
ref_ok = np.zeros(len(top1_orbi), dtype=bool)
for i, lid in enumerate(top1_orbi['library_wiki_id']):
    if lid in lib_peaks:
        ref_binned_orbi[i] = bin_spectrum(lib_peaks[lid])
        ref_ok[i] = True

print('Orbitrap top-1: %d spectra (%d with ref peaks)' % (len(top1_orbi), ref_ok.sum()))

# ── Compute NIST-model features for Orbitrap ──
# The NIST model expects: [emb_cosine, emb_dist, entropy_sim, delta_mda, n_candidates, esim_rank]
# For Orbitrap top-1, we compute ref embedding but have no query embedding (no raw query peaks).
# Workaround: use ref embedding self-similarity features + available scalar features.

encoder.eval()
with torch.no_grad():
    orbi_ref_emb = encoder(torch.tensor(ref_binned_orbi)).numpy()

# We don't have query embeddings, so emb_cosine/emb_dist between query-ref aren't available.
# Instead, use the ref embedding norm and entropy_similarity as the spectral match proxy.
# The NIST model was trained on query-candidate pairs, but for Orbitrap we only have the
# reference side. Use entropy_similarity as a bridge — it captures the query-ref relationship.

# Build feature matrix matching NIST feature columns
orbi_features = pd.DataFrame({
    'emb_cosine': top1_orbi['entropy_similarity'].values,  # proxy: esim correlates with emb_cosine
    'emb_dist': 1.0 - top1_orbi['entropy_similarity'].values,  # proxy: inverse
    'entropy_sim': top1_orbi['entropy_similarity'].values,
    'delta_mda': top1_orbi['delta_mda'].fillna(top1_orbi['delta_mda'].median()).values,
    'n_candidates': top1_orbi['n_candidates'].values,
    'esim_rank': np.ones(len(top1_orbi)),  # top-1 by definition
})
X_orbi_nist = orbi_features[NIST_FEAT_COLS].values.astype(np.float32)

# Apply NIST model → spectral confidence (no RT)
nist_confidence = gbm_nist_final.predict_proba(X_orbi_nist)[:, 1]
print('NIST spectral confidence — mean: %.3f  median: %.3f' % (
    nist_confidence.mean(), np.median(nist_confidence)))

# AUC of NIST model alone on Orbitrap (no RT yet)
auc_nist_alone = roc_auc_score(labels_orbi, nist_confidence)
print('NIST model AUC on Orbitrap (no RT): %.4f' % auc_nist_alone)

In [ ]:
# ── RT calibration (Option B): 500 Oliver spectra → thin calibration layer ──
# Split Oliver data: 500 for calibration, rest for evaluation
# Use stratified split to maintain class balance

np.random.seed(42)
n_cal = 500
cal_idx = np.sort(np.concatenate([
    np.random.choice(np.where(labels_orbi == 1)[0], n_cal * 7 // 10, replace=False),  # ~70% TP
    np.random.choice(np.where(labels_orbi == 0)[0], n_cal * 3 // 10, replace=False),  # ~30% FP
]))
eval_idx = np.array([i for i in range(len(labels_orbi)) if i not in set(cal_idx)])

print('Calibration set: %d (TP=%d, FP=%d)' % (
    len(cal_idx), labels_orbi[cal_idx].sum(), (labels_orbi[cal_idx]==0).sum()))
print('Evaluation set:  %d (TP=%d, FP=%d)' % (
    len(eval_idx), labels_orbi[eval_idx].sum(), (labels_orbi[eval_idx]==0).sum()))

# Calibration features: [nist_confidence, |delta_rt|, signed_delta_rt, sim_gap]
# These are the features NIST couldn't provide but Oliver's data has
delta_rt = top1_orbi['signed_delta_rt'].fillna(0).values
sim_gap = top1_orbi['sim_gap'].values

cal_features = np.column_stack([
    nist_confidence,
    np.abs(delta_rt),
    delta_rt,
    sim_gap,
])

# Logistic regression — thin calibration layer (4 features → P(correct))
# This is deliberately simple: we don't want to overfit on 500 samples
lr_cal = LogisticRegression(C=1.0, max_iter=1000)
lr_cal.fit(cal_features[cal_idx], labels_orbi[cal_idx])
final_scores = lr_cal.predict_proba(cal_features)[:, 1]

# ── Evaluation on held-out Oliver spectra ──
auc_nist_only = roc_auc_score(labels_orbi[eval_idx], nist_confidence[eval_idx])
auc_calibrated = roc_auc_score(labels_orbi[eval_idx], final_scores[eval_idx])
auc_esim_only = roc_auc_score(labels_orbi[eval_idx], top1_orbi.iloc[eval_idx]['entropy_similarity'].values)

print('\n=== Orbitrap Evaluation (held-out %d spectra) ===' % len(eval_idx))
print('  Entropy sim only:       AUC=%.4f' % auc_esim_only)
print('  NIST model (no RT):     AUC=%.4f' % auc_nist_only)
print('  NIST + RT calibration:  AUC=%.4f' % auc_calibrated)

# FDR comparison
print('\n%-30s  %8s %6s  %8s %6s' % ('Model', 'FDR@0.9', 'n', 'FDR@0.8', 'n'))
print('-' * 65)
for name, scores in [('Entropy sim', top1_orbi.iloc[eval_idx]['entropy_similarity'].values),
                     ('NIST model (no RT)', nist_confidence[eval_idx]),
                     ('NIST + RT calibrated', final_scores[eval_idx])]:
    lab = labels_orbi[eval_idx]
    parts = ['%-30s' % name]
    for t in [0.9, 0.8]:
        c = scores >= t; n = c.sum(); fp = ((lab==0)&c).sum()
        parts.append('%7.1f%% %6d' % (fp/n*100 if n>0 else 0, n))
    print('  '.join(parts))

# Calibration layer coefficients
print('\nCalibration layer coefficients:')
for feat, coef in zip(['nist_confidence', '|delta_rt|', 'signed_delta_rt', 'sim_gap'],
                       lr_cal.coef_[0]):
    print('  %-20s  %+.3f' % (feat, coef))
print('  intercept:           %+.3f' % lr_cal.intercept_[0])

## Summary plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lab_eval = labels_orbi[eval_idx]

# ── Panel 1: ROC ──
ax = axes[0]
models = [
    ('Entropy sim', top1_orbi.iloc[eval_idx]['entropy_similarity'].values, 'gray'),
    ('NIST (no RT)', nist_confidence[eval_idx], 'tab:blue'),
    ('NIST + RT cal', final_scores[eval_idx], 'tab:red'),
]
for name, scores, color in models:
    auc = roc_auc_score(lab_eval, scores)
    fpr, tpr, _ = roc_curve(lab_eval, scores)
    ax.plot(fpr, tpr, color=color, lw=2, label='%s (%.3f)' % (name, auc))
ax.plot([0,1],[0,1], 'k--', lw=0.8, alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC — Orbitrap held-out (%d spectra)' % len(eval_idx))
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
# Reference lines for previously computed models
ax.text(0.5, 0.3, 'GBM 12-feat: 0.882\nBayesian 3ch: 0.841',
        transform=ax.transAxes, fontsize=9, color='gray', style='italic')

# ── Panel 2: score distributions ──
ax = axes[1]
ax.hist(final_scores[eval_idx][lab_eval==1], bins=50, density=True, alpha=0.5,
        color='steelblue', label='TP')
ax.hist(final_scores[eval_idx][lab_eval==0], bins=50, density=True, alpha=0.5,
        color='salmon', label='FP')
ax.set_xlabel('NIST + RT calibrated score')
ax.set_ylabel('Density')
ax.set_title('Score distribution (held-out)')
ax.legend()

# ── Panel 3: NIST confidence vs RT effect ──
ax = axes[2]
ax.scatter(nist_confidence[eval_idx][lab_eval==1],
           np.abs(delta_rt[eval_idx][lab_eval==1]),
           s=5, alpha=0.3, color='steelblue', label='TP')
ax.scatter(nist_confidence[eval_idx][lab_eval==0],
           np.abs(delta_rt[eval_idx][lab_eval==0]),
           s=5, alpha=0.3, color='salmon', label='FP')
ax.set_xlabel('NIST spectral confidence (no RT)')
ax.set_ylabel('|delta_RT| (seconds)')
ax.set_title('Where does RT add value?\n(upper-right = NIST confident but RT is off)')
ax.legend()

plt.tight_layout()
plt.show()